In [1]:
# ============================================================
# DATASET: LendingClub
# Purpose: Cross-validation confirmation of the RF+SMOTE collapse
#          (currently based on a single train/test split only)
# ============================================================

import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

SCALED_DIR = Path("../Data/Processed")
FOLDS_DIR = Path("../logs/fold_assignments")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

# Load the LendingClub training data (folds were generated on this exact set)
X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
clean_column_names(X_lending_train)
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()

# Load the previously saved fold indices, so we reuse the exact same folds
# generated back in Step 2, rather than creating new ones
with open(FOLDS_DIR / "lending_folds.json") as f:
    lending_folds = json.load(f)

print(X_lending_train.shape, y_lending_train.shape)
print(f"Number of folds loaded: {len(lending_folds)}")

(8000, 69) (8000,)
Number of folds loaded: 5


In [2]:
# ============================================================
# DATASET: LendingClub
# Step: RF with/without SMOTE, evaluated on each of the 5 saved folds
# ============================================================

fold_results = []

for fold_name, indices in lending_folds.items():
    train_idx = indices["train_idx"]
    val_idx = indices["val_idx"]

    # Split this fold's training portion from its validation portion
    X_fold_train = X_lending_train.iloc[train_idx]
    y_fold_train = y_lending_train.iloc[train_idx]
    X_fold_val = X_lending_train.iloc[val_idx]
    y_fold_val = y_lending_train.iloc[val_idx]

    # RF without SMOTE
    rf_plain = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200)
    rf_plain.fit(X_fold_train, y_fold_train)
    y_pred_plain = rf_plain.predict(X_fold_val)
    y_proba_plain = rf_plain.predict_proba(X_fold_val)[:, 1]

    fold_results.append({
        "fold": fold_name,
        "config": "RF (no SMOTE)",
        "AUC-ROC": roc_auc_score(y_fold_val, y_proba_plain),
        "F1": f1_score(y_fold_val, y_pred_plain),
        "G-mean": geometric_mean_score(y_fold_val, y_pred_plain),
        "MCC": matthews_corrcoef(y_fold_val, y_pred_plain),
        "positive_predictions": int(y_pred_plain.sum())  # tracking this specifically,
        # since the original finding was that RF predicted ZERO positives after SMOTE
    })

    # RF with SMOTE - applied only within this fold's training portion
    rf_smote = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_SEED)),
        ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
    ])
    rf_smote.fit(X_fold_train, y_fold_train)
    y_pred_smote = rf_smote.predict(X_fold_val)
    y_proba_smote = rf_smote.predict_proba(X_fold_val)[:, 1]

    fold_results.append({
        "fold": fold_name,
        "config": "RF (with SMOTE)",
        "AUC-ROC": roc_auc_score(y_fold_val, y_proba_smote),
        "F1": f1_score(y_fold_val, y_pred_smote),
        "G-mean": geometric_mean_score(y_fold_val, y_pred_smote),
        "MCC": matthews_corrcoef(y_fold_val, y_pred_smote),
        "positive_predictions": int(y_pred_smote.sum())
    })

    print(f"{fold_name} done")

fold_results_df = pd.DataFrame(fold_results)
print(fold_results_df)

fold_0 done
fold_1 done
fold_2 done
fold_3 done
fold_4 done
     fold           config   AUC-ROC        F1    G-mean       MCC  \
0  fold_0    RF (no SMOTE)  0.749830  0.193548  0.327327  0.324755   
1  fold_0  RF (with SMOTE)  0.787021  0.187500  0.327223  0.279664   
2  fold_1    RF (no SMOTE)  0.760189  0.133333  0.267261  0.265078   
3  fold_1  RF (with SMOTE)  0.805400  0.000000  0.000000 -0.003338   
4  fold_2    RF (no SMOTE)  0.784147  0.068966  0.188982  0.187380   
5  fold_2  RF (with SMOTE)  0.840649  0.000000  0.000000  0.000000   
6  fold_3    RF (no SMOTE)  0.727474  0.066667  0.185695  0.184062   
7  fold_3  RF (with SMOTE)  0.790678  0.000000  0.000000  0.000000   
8  fold_4    RF (no SMOTE)  0.841744  0.129032  0.262613  0.260385   
9  fold_4  RF (with SMOTE)  0.827520  0.000000  0.000000 -0.003398   

   positive_predictions  
0                     3  
1                     4  
2                     2  
3                     1  
4                     1  
5            

In [3]:
# ============================================================
# DATASET: German Credit + LendingClub
# Purpose: CV confirmation of the SHAP stability ordering (finding #6)
#          Using 3 of the 5 saved folds to keep SHAP compute manageable
# ============================================================

import pandas as pd
import numpy as np
import json
import re
import shap
import gc
from pathlib import Path
from scipy.stats import spearmanr
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

SCALED_DIR = Path("../Data/Processed")
FOLDS_DIR = Path("../logs/fold_assignments")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

# German Credit
X_german_train = pd.read_csv(SCALED_DIR / "german_X_train.csv")
clean_column_names(X_german_train)
y_german_train = pd.read_csv(SCALED_DIR / "german_y_train.csv").squeeze()

with open(FOLDS_DIR / "german_folds.json") as f:
    german_folds = json.load(f)

# LendingClub
X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
clean_column_names(X_lending_train)
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()

with open(FOLDS_DIR / "lending_folds.json") as f:
    lending_folds = json.load(f)

print("German:", X_german_train.shape, "folds:", len(german_folds))
print("Lending:", X_lending_train.shape, "folds:", len(lending_folds))

c:\Users\Thokozani\Desktop\BcomHons\Second Semester\Information Systems Research\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


German: (800, 40) folds: 5
Lending: (8000, 69) folds: 5


In [4]:
# ============================================================
# Reusable function: fit XGBoost with/without SMOTE on a fold,
# compute SHAP importance rankings, return Spearman correlation
# ============================================================

def shap_stability_for_fold(X_train_full, y_train_full, train_idx, val_idx, sample_size=None):
    # Split this fold's training portion from its validation portion
    X_fold_train = X_train_full.iloc[train_idx]
    y_fold_train = y_train_full.iloc[train_idx]
    X_fold_val = X_train_full.iloc[val_idx]

    # Optionally reduce the SHAP evaluation sample (used for Home Credit later,
    # to manage memory - not needed for the smaller datasets)
    if sample_size and len(X_fold_val) > sample_size:
        X_fold_val = X_fold_val.sample(n=sample_size, random_state=RANDOM_SEED)

    # No SMOTE
    xgb_no_smote = XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss")
    xgb_no_smote.fit(X_fold_train, y_fold_train)
    explainer_no_smote = shap.TreeExplainer(xgb_no_smote)
    shap_no_smote = explainer_no_smote.shap_values(X_fold_val)
    importance_no_smote = pd.Series(
        abs(shap_no_smote).mean(axis=0), index=X_fold_val.columns
    ).sort_values(ascending=False)
    del xgb_no_smote, explainer_no_smote, shap_no_smote
    gc.collect()

    # With SMOTE
    xgb_smote_pipe = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_SEED)),
        ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
    ])
    xgb_smote_pipe.fit(X_fold_train, y_fold_train)
    xgb_smote_model = xgb_smote_pipe.named_steps["clf"]
    explainer_smote = shap.TreeExplainer(xgb_smote_model)
    shap_smote = explainer_smote.shap_values(X_fold_val)
    importance_smote = pd.Series(
        abs(shap_smote).mean(axis=0), index=X_fold_val.columns
    ).sort_values(ascending=False)
    del xgb_smote_pipe, xgb_smote_model, explainer_smote, shap_smote
    gc.collect()

    # Spearman correlation between the two rankings for this fold
    corr, p = spearmanr(
        importance_smote.rank(),
        importance_no_smote.reindex(importance_smote.index).rank()
    )
    return corr, p

In [5]:
# ============================================================
# Run the stability check on fold_0, fold_1, fold_2 only (3 of 5) -
# enough to confirm the single-split result isn't a fluke, without
# the full computational cost of all 5 folds x 2 models x 2 datasets
# ============================================================

fold_subset = ["fold_0", "fold_1", "fold_2"]
stability_results = []

for fold_name in fold_subset:
    train_idx = german_folds[fold_name]["train_idx"]
    val_idx = german_folds[fold_name]["val_idx"]
    corr, p = shap_stability_for_fold(X_german_train, y_german_train, train_idx, val_idx)
    stability_results.append({"dataset": "German Credit", "fold": fold_name, "spearman": corr, "p_value": p})
    print(f"German Credit {fold_name}: Spearman = {corr:.4f}")

for fold_name in fold_subset:
    train_idx = lending_folds[fold_name]["train_idx"]
    val_idx = lending_folds[fold_name]["val_idx"]
    corr, p = shap_stability_for_fold(X_lending_train, y_lending_train, train_idx, val_idx)
    stability_results.append({"dataset": "LendingClub", "fold": fold_name, "spearman": corr, "p_value": p})
    print(f"LendingClub {fold_name}: Spearman = {corr:.4f}")

stability_df = pd.DataFrame(stability_results)
print(stability_df)

German Credit fold_0: Spearman = 0.9753
German Credit fold_1: Spearman = 0.9843
German Credit fold_2: Spearman = 0.9656
LendingClub fold_0: Spearman = 0.7673
LendingClub fold_1: Spearman = 0.7682
LendingClub fold_2: Spearman = 0.7733
         dataset    fold  spearman       p_value
0  German Credit  fold_0  0.975312  1.562253e-26
1  German Credit  fold_1  0.984324  3.019738e-30
2  German Credit  fold_2  0.965628  7.728663e-24
3    LendingClub  fold_0  0.767281  1.484708e-14
4    LendingClub  fold_1  0.768213  1.319625e-14
5    LendingClub  fold_2  0.773294  6.870028e-15


In [6]:
# ============================================================
# DATASET: Home Credit
# Purpose: CV confirmation of the SHAP stability ordering (finding #6)
#          Using 2 of the 5 saved folds, with the memory-safe sequential
#          fit/explain pattern that resolved the original MemoryError
# ============================================================

X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
clean_column_names(X_home_train)
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()

with open(FOLDS_DIR / "home_folds.json") as f:
    home_folds = json.load(f)

print(X_home_train.shape, "folds:", len(home_folds))

(246005, 100) folds: 5


In [7]:
# ============================================================
# DATASET: Home Credit
# Step: SHAP stability check across 2 folds, with a reduced
# validation sample (300 rows) to manage memory, matching the
# approach that worked for the original single-split Home Credit run
# ============================================================

home_fold_subset = ["fold_0", "fold_1"]
home_stability_results = []

for fold_name in home_fold_subset:
    train_idx = home_folds[fold_name]["train_idx"]
    val_idx = home_folds[fold_name]["val_idx"]

    corr, p = shap_stability_for_fold(
        X_home_train, y_home_train, train_idx, val_idx, sample_size=300
    )
    home_stability_results.append({"dataset": "Home Credit", "fold": fold_name, "spearman": corr, "p_value": p})
    print(f"Home Credit {fold_name}: Spearman = {corr:.4f}")

home_stability_df = pd.DataFrame(home_stability_results)
print(home_stability_df)

Home Credit fold_0: Spearman = 0.9256
Home Credit fold_1: Spearman = 0.9150
       dataset    fold  spearman       p_value
0  Home Credit  fold_0  0.925612  3.851258e-43
1  Home Credit  fold_1  0.915043  1.997194e-40
